In [50]:
import duckdb
from pathlib import Path

DB_PATH = Path("../mimic4_note.db").resolve()

con = duckdb.connect(str(DB_PATH))

In [51]:
nlp_subset_readmit = con.sql("""
SELECT *
FROM nlp_subset_readmit
""").df()

Because ClinicalBERT has a 512-token input limit, longer discharge sections were truncated during embedding generation. To reduce information loss, the analysis used clinically targeted sections rather than the beginning of the full note.

In [26]:
import re

def extract_section(text, start_header, end_headers):
    if not end_headers:
        pattern = re.compile(
            rf"{re.escape(start_header)}\s*(.*)$",
            flags=re.IGNORECASE | re.DOTALL
        )
    else:
        pattern = re.compile(
            rf"{re.escape(start_header)}\s*(.*?)(?=\n\s*(?:{'|'.join(map(re.escape, end_headers))})|$)",
            flags=re.IGNORECASE | re.DOTALL
        )

    match = pattern.search(text)
    return match.group(1).strip() if match else ""

## Brief Hospital Course

In [27]:
def extract_brief_hospital_course(text):
    return extract_section(
        text,
        "Brief Hospital Course:",
        [
            "Medications on Admission:",
            "Discharge Medications:",
            "Discharge Disposition:",
            "Discharge Diagnosis:",
            "Discharge Condition:",
            "Discharge Instructions:",
            "Followup Instructions:"
        ]
    )

## Medication Reconciliation (changes in medications from admission to discharge) 

This captures BOTH admission and discharge meds.

In [28]:
def extract_medication_reconciliation(text):
    meds_on_admission = extract_section(
        text,
        "Medications on Admission:",
        [
            "Discharge Medications:"
        ]
    )

    discharge_meds = extract_section(
        text,
        "Discharge Medications:",
        [
            "Discharge Disposition:",
            "Discharge Diagnosis:",
            "Discharge Condition:",
            "Discharge Instructions:",
            "Followup Instructions:"
        ]
    )

    combined = "\n\n".join([
        "MEDICATIONS ON ADMISSION",
        meds_on_admission,
        "",
        "DISCHARGE MEDICATIONS",
        discharge_meds
    ])

    return combined.strip()

## Discharge Instructions + Followup Instructions

In [29]:
def extract_discharge_planning(text):
    discharge_instructions = extract_section(
        text,
        "Discharge Instructions:",
        [
            "Followup Instructions:"
        ]
    )

    followup = extract_section(
        text,
        "Followup Instructions:",
        []
    )

    combined = "\n\n".join([
        "DISCHARGE INSTRUCTIONS",
        discharge_instructions,
        "",
        "FOLLOWUP INSTRUCTIONS",
        followup
    ])

    return combined.strip()

## Create Columns

In [43]:
nlp_subset_readmit_sectioned = nlp_subset_readmit.copy()

nlp_subset_readmit_sectioned["brief_hospital_course"] = (
    nlp_subset_readmit_sectioned["text"]
    .apply(extract_brief_hospital_course)
)

nlp_subset_readmit_sectioned["medication_reconciliation"] = (
    nlp_subset_readmit_sectioned["text"]
    .apply(extract_medication_reconciliation)
)

nlp_subset_readmit_sectioned["discharge_planning"] = (
    nlp_subset_readmit_sectioned["text"]
    .apply(extract_discharge_planning)
)

Full text is found at example_note.txt

## Ensure the correct section of the text is being extracted

In [44]:
idx = 0

print(" ".join(str(nlp_subset_readmit_sectioned.loc[idx, "brief_hospital_course"]).split()[:50]), end="\n\n")
print(" ".join(str(nlp_subset_readmit_sectioned.loc[idx, "medication_reconciliation"]).split()[:50]), end="\n\n")
print(" ".join(str(nlp_subset_readmit_sectioned.loc[idx, "discharge_planning"]).split()[:50]), end="\n\n")

Mrs. ___ is a ___ community-dwelling woman w/ HFpEF, HTN, CKD III, DM2, secondary adrenal insufficiency (on prednisone), and recurrent gout with two prior admission this year for gout flares, who presents with fever and ankle pain - likely consistent with another gout flare. #OLIGOARTICULAR GOUT FLARE #FEVERS (resolved) #ANKLE

MEDICATIONS ON ADMISSION The Preadmission Medication list is accurate and complete. 1. Acetaminophen 1000 mg PO Q8H:PRN Pain - Mild/Fever 2. Albuterol Inhaler 1 PUFF IH Q6H:PRN dyspnea 3. Allopurinol ___ mg PO DAILY 4. Aspirin 81 mg PO DAILY 5. Docusate Sodium 100 mg PO DAILY 6. Fluticasone Propionate

DISCHARGE INSTRUCTIONS Dear Ms. ___, You were admitted to the hospital because of a gout flare in your ankle joints. We increased your allopurinol to 200 mg daily and started you on a preventative medication called colchicine at 0.6 mg daily. We also started you on a prednisone taper. While



In [45]:
nlp_subset_readmit_sectioned.columns

Index(['note_id', 'subject_id', 'hadm_id', 'charttime', 'storetime', 'text',
       'days_to_next_admission', 'readmit_30d', 'brief_hospital_course',
       'medication_reconciliation', 'discharge_planning'],
      dtype='str')

In [53]:
# duckdb doesn't support string types, so we need to cast the columns to object before registering the dataframe as a table; converted to varchar in duckdb
text_cols = [
    "note_id",
    "text",
    "brief_hospital_course",
    "medication_reconciliation",
    "discharge_planning",
    ]

tmp = nlp_subset_readmit_sectioned.copy()

for col in text_cols:
    tmp[col] = tmp[col].astype(object)

con.register("tmp_df", tmp)

con.sql("""
CREATE OR REPLACE TABLE nlp_subset_readmit_sectioned AS
SELECT
    CAST(note_id AS VARCHAR) AS note_id,
    CAST(subject_id AS INTEGER) AS subject_id,
    CAST(hadm_id AS INTEGER) AS hadm_id,
    CAST(charttime AS TIMESTAMP) AS charttime,
    CAST(storetime AS TIMESTAMP) AS storetime,
    CAST(text AS VARCHAR) AS text,
    CAST(days_to_next_admission AS INTEGER) AS days_to_next_admission,
    CAST(readmit_30d AS SMALLINT) AS readmit_30d,
    CAST(brief_hospital_course AS VARCHAR) AS brief_hospital_course,
    CAST(medication_reconciliation AS VARCHAR) AS medication_reconciliation,
    CAST(discharge_planning AS VARCHAR) AS discharge_planning,
FROM tmp_df
""")

In [54]:
con.sql("DESCRIBE nlp_subset_readmit_sectioned").df()

,column_name,column_type,null,key,default,extra
0,note_id,VARCHAR,YES,None,None,None
1,subject_id,INTEGER,YES,None,None,None
2,hadm_id,INTEGER,YES,None,None,None
3,charttime,TIMESTAMP,YES,None,None,None
4,storetime,TIMESTAMP,YES,None,None,None
5,text,VARCHAR,YES,None,None,None
6,days_to_next_admission,INTEGER,YES,None,None,None
7,readmit_30d,SMALLINT,YES,None,None,None
8,brief_hospital_course,VARCHAR,YES,None,None,None
9,medication_reconciliation,VARCHAR,YES,None,None,None


## Testing Group Stratified Train/Test Split

In [21]:
from sklearn.model_selection import StratifiedGroupKFold

X = nlp_subset_readmit.index
y = nlp_subset_readmit["readmit_30d"]
groups = nlp_subset_readmit["subject_id"]

sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=321
)

train_idx, test_idx = next(sgkf.split(X, y, groups))

train_df = nlp_subset_readmit.iloc[train_idx].copy()
test_df = nlp_subset_readmit.iloc[test_idx].copy()

In [22]:
set(train_df["subject_id"]) & set(test_df["subject_id"])

set()

# Proof of concept code for using the huggingface pipeline

In [ ]:
import os
from transformers import AutoTokenizer, AutoModel

# huggingface-cli login within the terminal (need a token)

In [11]:
tokenizer = AutoTokenizer.from_pretrained(
    "medicalai/ClinicalBERT"
)

model = AutoModel.from_pretrained(
    "medicalai/ClinicalBERT"
)

pytorch_model.bin:   0%|          | 0.00/542M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: medicalai/ClinicalBERT
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/542M [00:00<?, ?B/s]

In [ ]:
find ~/.cache/huggingface -name "*.bin" -o -name "*.safetensors"

In [13]:
print(model.name_or_path)

medicalai/ClinicalBERT


## Testing the Model on One Note

In [24]:
import torch

# move model to GPU if available

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)
model.eval()

text = nlp_subset_readmit.loc[0, "text"]

tokens = tokenizer(
    text,
    truncation=True,
    padding="max_length",
    max_length=512,
    return_tensors="pt"
)

tokens = {k: v.to(device) for k, v in tokens.items()}

with torch.no_grad():
    outputs = model(**tokens)

print(outputs.last_hidden_state.shape)

torch.Size([1, 512, 768])


In [25]:
embedding = outputs.last_hidden_state[:, 0, :].cpu().numpy()

print(embedding.shape)

(1, 768)


* 1 note
* 512 tokens
* 768 embedding dimensions per token

In [14]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)
model.eval()

DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(119547, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSelfAttention(
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=True)
          (lin2): 

In [ ]:
con.close()